# Model 3 — model comparison, final training and test (scored at carrier handoff)

**Upload to the Colab file panel before running:**
* `orders_clean.csv` (required)
* `model3_selected_features.json` (from the feature-selection notebook; if missing, the same 10 columns are used from a built-in list)
* `model1_production_test_predictions.csv` (optional — from the production Model 1 zip; enables the Model 1 vs Model 3 comparison on the same test orders)

Then *Runtime → Run all*. Expect about 15–25 minutes.

**Plan**

| Step | Data used | What happens |
|---|---|---|
| 1. Compare 4 model families | Training period only (4 time-ordered folds, Jan 2017 – Mar 2018) | Logistic regression, XGBoost, LightGBM, CatBoost; a small settings grid each; scored on mean PR-AUC |
| 2. Choose — **locked before the test set is opened** | Fold results | XGBoost is kept unless another family beats it by more than 2% **and** wins at least 3 of 4 folds |
| 3. Final fit | Whole training period | Chosen model + settings |
| 4. Calibrate | April 2018 (late + on-time) | Alert threshold (10% FPR on April on-time orders) and real probabilities |
| 5. Test, once | May – Aug 2018 | Final numbers, side by side with Model 1 |

In [ ]:
# ---------------------------------------------------------------- setup (installs CatBoost if Colab lacks it)
import importlib.util, subprocess, sys
for pkg in ["catboost", "lightgbm"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, os, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn, xgboost, lightgbm, catboost
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
try:
    display
except NameError:
    display = print

SEED = 42
VERSIONS = {"scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__,
            "lightgbm": lightgbm.__version__, "catboost": catboost.__version__}
print(VERSIONS)
print("Production Model 1 used scikit-learn 1.6.1 and xgboost 3.4.1")

In [ ]:
# ---------------------------------------------------------------- load data and the selected columns
def find(name):
    for d in [".", "/content", "/content/drive/MyDrive"]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    return None

CLEAN_PATH = find("orders_clean.csv")
assert CLEAN_PATH, "Upload orders_clean.csv to the Colab file panel first."
clean = pd.read_csv(CLEAN_PATH, parse_dates=["purchase_ts"]).sort_values("purchase_ts").reset_index(drop=True)
print(f"Loaded {CLEAN_PATH}: {len(clean):,} rows x {clean.shape[1]} columns (expected 94,930 x 39)")
bad = clean["is_late"].isna() | clean["purchase_ts"].isna()
if bad.any():
    print(f"WARNING: dropping {bad.sum()} row(s) with no label/date — re-upload the file if this is more than 1-2.")
    clean = clean[~bad].reset_index(drop=True)
clean["is_late"] = clean["is_late"].astype(int)

SEL_PATH = find("model3_selected_features.json")
if SEL_PATH:
    sel = json.load(open(SEL_PATH))
    FEATURES, CATEGORICAL = sel["selected_features"], sel["selected_categorical"]
    print("Columns from", SEL_PATH)
else:
    FEATURES = ["promise_days", "weight_g", "customer_state", "dist_km", "route_recent_days",
                "promise_buffer_route", "promise_buffer_state", "handling_days", "seller_overrun_days",
                "post_approval_days"]
    CATEGORICAL = ["customer_state"]
    print("model3_selected_features.json not found — using the built-in 10-column list")
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]
print(f"{len(FEATURES)} columns:", FEATURES)

P = clean.purchase_ts
TRAIN = clean[(P >= "2017-01-01") & (P < "2018-04-01")].reset_index(drop=True)
CALIB = clean[(P >= "2018-04-01") & (P < "2018-05-01")].reset_index(drop=True)
TEST = clean[(P >= "2018-05-01") & (P < "2018-09-01")].reset_index(drop=True)
del P
for name, d in [("train", TRAIN), ("April calib", CALIB), ("test", TEST)]:
    print(f"{name:12s}: {len(d):6,} orders | {d.is_late.sum():5,} late ({d.is_late.mean():.1%})")
assert len(TEST) == 24928 and TEST.is_late.sum() == 1114, "test set differs from Model 1's — check the file"

## Shared preparation

Every model receives the **same** prepared table, so the Kafka scorer needs one `prepare_features()` no matter which model wins:
* `customer_state` → integer code learned on training data; unseen states → −1
* numbers left as they are; missing values stay NaN

Tree models handle NaN and the integer state code directly (LightGBM and CatBoost are told it is a category). Logistic regression gets its own standard scikit-learn pipeline inside the model: fill missing with the median + a "was missing" flag, scale, one-hot the state.

In [ ]:
def build_codes(train_df):
    return {c: {v: i for i, v in enumerate(sorted(train_df[c].astype(str).unique()))} for c in CATEGORICAL}

def prepare(df, codes):
    X = df[FEATURES].copy()
    for c in CATEGORICAL:
        X[c] = X[c].astype(str).map(codes[c]).fillna(-1).astype(int)
    for c in NUMERIC:
        X[c] = X[c].astype(float)
    return X

def make_model(family, params, pos_weight):
    cat_idx = [FEATURES.index(c) for c in CATEGORICAL]
    if family == "LogReg":
        pre = ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMERIC),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), CATEGORICAL)])
        return make_pipeline(pre, LogisticRegression(C=params["C"], class_weight={0: 1.0, 1: pos_weight},
                                                     max_iter=3000))
    if family == "XGBoost":
        return XGBClassifier(**params, objective="binary:logistic", eval_metric="logloss", tree_method="hist",
                             scale_pos_weight=pos_weight, random_state=SEED, n_jobs=-1, verbosity=0)
    if family == "LightGBM":
        return LGBMClassifier(**params, scale_pos_weight=pos_weight, random_state=SEED, n_jobs=-1, verbose=-1)
    if family == "CatBoost":
        return CatBoostClassifier(**params, loss_function="Logloss", class_weights=[1.0, pos_weight],
                                  cat_features=cat_idx, random_seed=SEED, verbose=False,
                                  allow_writing_files=False, thread_count=-1)
    raise ValueError(family)

def fit_model(family, params, X, y):
    pos_weight = (y == 0).sum() / max((y == 1).sum(), 1) * WEIGHT_MULTIPLIER
    m = make_model(family, params, pos_weight)
    if family == "LightGBM":
        m.fit(X, y, categorical_feature=CATEGORICAL)
    else:
        m.fit(X, y)
    return m

WEIGHT_MULTIPLIER = 0.75   # same as production Model 1

## Step 1 — compare the four families on time-ordered folds

Same 4 folds as the feature-selection notebook (train before a cut-off, skip one month, validate on the next two). Each family gets a small grid of sensible settings; the best setting per family is kept.

In [ ]:
FOLDS = [("2017-07-01", "2017-08-01", "2017-10-01"),
         ("2017-09-01", "2017-10-01", "2017-12-01"),
         ("2017-11-01", "2017-12-01", "2018-02-01"),
         ("2018-01-01", "2018-02-01", "2018-04-01")]
fold_data = []
for tr_end, va_start, va_end in FOLDS:
    tr = TRAIN[TRAIN.purchase_ts < tr_end]
    va = TRAIN[(TRAIN.purchase_ts >= va_start) & (TRAIN.purchase_ts < va_end)]
    codes = build_codes(tr)
    fold_data.append((prepare(tr, codes), tr.is_late.values, prepare(va, codes), va.is_late.values))

GRID = {
    "LogReg":   [{"C": 0.001}, {"C": 0.003}, {"C": 0.01}, {"C": 0.1}, {"C": 1.0}],
    "XGBoost":  [dict(n_estimators=600, max_depth=2, learning_rate=0.03, min_child_weight=10, subsample=0.9, colsample_bytree=0.9, reg_lambda=10.0),
                 dict(n_estimators=500, max_depth=3, learning_rate=0.03, min_child_weight=10, subsample=0.9, colsample_bytree=0.9, reg_lambda=10.0),
                 dict(n_estimators=400, max_depth=4, learning_rate=0.03, min_child_weight=20, subsample=0.9, colsample_bytree=0.9, reg_lambda=10.0),
                 dict(n_estimators=800, max_depth=3, learning_rate=0.02, min_child_weight=30, subsample=0.8, colsample_bytree=0.8, reg_lambda=20.0)],
    "LightGBM": [dict(n_estimators=500, learning_rate=0.03, num_leaves=7,  min_child_samples=50,  subsample=0.9, subsample_freq=1, colsample_bytree=0.9, reg_lambda=10.0),
                 dict(n_estimators=500, learning_rate=0.03, num_leaves=15, min_child_samples=50,  subsample=0.9, subsample_freq=1, colsample_bytree=0.9, reg_lambda=10.0),
                 dict(n_estimators=400, learning_rate=0.03, num_leaves=31, min_child_samples=100, subsample=0.9, subsample_freq=1, colsample_bytree=0.9, reg_lambda=10.0),
                 dict(n_estimators=800, learning_rate=0.02, num_leaves=15, min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=20.0)],
    "CatBoost": [dict(iterations=700, depth=4, learning_rate=0.03, l2_leaf_reg=8.0),
                 dict(iterations=500, depth=6, learning_rate=0.03, l2_leaf_reg=8.0),
                 dict(iterations=900, depth=5, learning_rate=0.02, l2_leaf_reg=15.0)],
}

def recall_top(y, s, k=0.10):
    n = max(1, int(len(y) * k)); return y[np.argsort(-s)[:n]].sum() / max(y.sum(), 1)

rows = []
t0 = time.time()
for family, configs in GRID.items():
    for i, params in enumerate(configs):
        pr, roc, rt = [], [], []
        for Xtr, ytr, Xva, yva in fold_data:
            s = fit_model(family, params, Xtr, ytr).predict_proba(Xva)[:, 1]
            pr.append(average_precision_score(yva, s)); roc.append(roc_auc_score(yva, s)); rt.append(recall_top(yva, s))
        rows.append(dict(family=family, config=i, params=params, pr_auc=np.mean(pr), pr_auc_std=np.std(pr, ddof=1),
                         roc_auc=np.mean(roc), recall_top10=np.mean(rt), fold_pr_auc=[round(v, 4) for v in pr]))
        print(f"{family:9s} config {i}: PR-AUC {np.mean(pr):.4f} ± {np.std(pr, ddof=1):.4f} | ROC {np.mean(roc):.4f} "
              f"| folds {[round(v, 3) for v in pr]}  ({time.time() - t0:.0f}s)")
cv = pd.DataFrame(rows)

In [ ]:
best_per_family = cv.loc[cv.groupby("family").pr_auc.idxmax()].sort_values("pr_auc", ascending=False).reset_index(drop=True)
display(best_per_family[["family", "config", "pr_auc", "pr_auc_std", "roc_auc", "recall_top10", "fold_pr_auc"]].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
for _, r in best_per_family.iterrows():
    ax.plot(range(1, 5), r.fold_pr_auc, marker="o", label=f"{r.family} (mean {r.pr_auc:.3f})")
ax.set_xticks(range(1, 5)); ax.set_xticklabels(["Aug–Sep 17", "Oct–Nov 17", "Dec 17–Jan 18", "Feb–Mar 18"])
ax.set_ylabel("PR-AUC"); ax.set_title("Best setting per family, fold by fold"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Step 2 — choose the model (locked before the test set is opened)

Rule written down in advance: **keep XGBoost** (one library for both models in the Kafka scorer) **unless** another family's mean PR-AUC is more than 2% higher **and** it beats XGBoost in at least 3 of the 4 folds. Logistic regression is the baseline that shows how much the trees add.

In [ ]:
xgb_row = best_per_family[best_per_family.family == "XGBoost"].iloc[0]
choice = xgb_row
for _, r in best_per_family.iterrows():
    if r.family == "XGBoost":
        continue
    wins = sum(a > b for a, b in zip(r.fold_pr_auc, xgb_row.fold_pr_auc))
    margin = r.pr_auc / xgb_row.pr_auc - 1
    print(f"{r.family:9s} vs XGBoost: mean PR-AUC {margin:+.1%}, wins {wins}/4 folds")
    if margin > 0.02 and wins >= 3 and r.pr_auc > choice.pr_auc:
        choice = r

CHOSEN_FAMILY, CHOSEN_PARAMS = choice.family, choice.params
logreg = best_per_family[best_per_family.family == "LogReg"].iloc[0]
print(f"\nCHOSEN: {CHOSEN_FAMILY} config {choice.config}  (CV PR-AUC {choice.pr_auc:.4f}, ROC {choice.roc_auc:.4f})")
print(f"Gain over the logistic-regression baseline: PR-AUC {choice.pr_auc - logreg.pr_auc:+.4f}, ROC {choice.roc_auc - logreg.roc_auc:+.4f}")
print("Settings:", CHOSEN_PARAMS)

## Steps 3–4 — final fit on the whole training period, calibrate on April

All four families are refit too, but **only for the report table** — the choice above is already fixed.

In [ ]:
codes = build_codes(TRAIN)
X_tr, X_ca, X_te = prepare(TRAIN, codes), prepare(CALIB, codes), prepare(TEST, codes)
y_tr, y_ca, y_te = TRAIN.is_late.values, CALIB.is_late.values, TEST.is_late.values

final = {}
for _, r in best_per_family.iterrows():
    m = fit_model(r.family, r.params, X_tr, y_tr)
    final[r.family] = dict(model=m, raw_ca=m.predict_proba(X_ca)[:, 1], raw_te=m.predict_proba(X_te)[:, 1])

FPR_TARGET = 0.10
def at_threshold(y, s, t):
    a = (s >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, a, labels=[0, 1]).ravel()
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(threshold=float(t), alert_rate=float(a.mean()), fpr=fp / max(fp + tn, 1), precision=prec, recall=rec,
                f1=2 * prec * rec / max(prec + rec, 1e-9), TP=int(tp), FP=int(fp), TN=int(tn), FN=int(fn))
def top_k(y, s, k):
    n = max(1, int(len(y) * k)); tp = int(y[np.argsort(-s)[:n]].sum())
    return dict(alerts=n, TP=tp, precision=tp / n, recall=tp / y.sum())

report = []
for fam, d in final.items():
    t = float(np.quantile(d["raw_ca"][y_ca == 0], 1 - FPR_TARGET))
    d["threshold"] = t
    d["calibrator"] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(d["raw_ca"], y_ca)
    d["prob_te"] = d["calibrator"].predict(d["raw_te"])
    th = at_threshold(y_te, d["raw_te"], t)
    report.append(dict(family=fam, chosen=(fam == CHOSEN_FAMILY),
                       april_pr_auc=average_precision_score(y_ca, d["raw_ca"]), april_roc_auc=roc_auc_score(y_ca, d["raw_ca"]),
                       test_pr_auc=average_precision_score(y_te, d["raw_te"]), test_roc_auc=roc_auc_score(y_te, d["raw_te"]),
                       test_recall_top10=top_k(y_te, d["raw_te"], 0.10)["recall"],
                       thr=t, test_fpr=th["fpr"], test_recall=th["recall"], test_precision=th["precision"], TP=th["TP"], FP=th["FP"]))
report = pd.DataFrame(report).sort_values("chosen", ascending=False)
print("April = calibration month; test = May–Aug 2018. Choice was fixed in step 2.")
display(report.round(4))

## Step 5 — the chosen Model 3 on the test set, and Model 1 vs Model 3 on the same orders

In [ ]:
d = final[CHOSEN_FAMILY]
test_metrics = dict(
    pr_auc=average_precision_score(y_te, d["raw_te"]), roc_auc=roc_auc_score(y_te, d["raw_te"]), base_rate=float(y_te.mean()),
    at_threshold=at_threshold(y_te, d["raw_te"], d["threshold"]),
    top5=top_k(y_te, d["raw_te"], 0.05), top10=top_k(y_te, d["raw_te"], 0.10), top20=top_k(y_te, d["raw_te"], 0.20),
    mean_calibrated_prob=float(d["prob_te"].mean()))
cal_metrics = dict(pr_auc=average_precision_score(y_ca, d["raw_ca"]), roc_auc=roc_auc_score(y_ca, d["raw_ca"]))

t = test_metrics["at_threshold"]
print(f"Model 3 ({CHOSEN_FAMILY}) on May–Aug 2018: PR-AUC {test_metrics['pr_auc']:.4f} | ROC-AUC {test_metrics['roc_auc']:.4f}")
print(f"  threshold {d['threshold']:.4f}: FPR {t['fpr']:.1%}, alerts {t['alert_rate']:.1%}, recall {t['recall']:.1%}, "
      f"precision {t['precision']:.1%} (TP {t['TP']}, FP {t['FP']})")
for k in ["top5", "top10", "top20"]:
    r = test_metrics[k]
    print(f"  riskiest {k[3:]:>2}%: {r['TP']} late caught | recall {r['recall']:.1%} | precision {r['precision']:.1%}")
print(f"  mean calibrated probability {test_metrics['mean_calibrated_prob']:.3f} vs true late rate {y_te.mean():.3f}")

M1_PATH = find("model1_production_test_predictions.csv")
if M1_PATH:
    m1 = pd.read_csv(M1_PATH)
    both = pd.DataFrame({"order_id": TEST.order_id, "y": y_te, "m3": d["raw_te"]}).merge(
        m1[["order_id", "raw_score"]].rename(columns={"raw_score": "m1"}), on="order_id")
    yb = both.y.values
    rng = np.random.default_rng(SEED); diffs = []
    for _ in range(1000):
        i = rng.integers(0, len(yb), len(yb))
        diffs.append(average_precision_score(yb[i], both.m3.values[i]) - average_precision_score(yb[i], both.m1.values[i]))
    diffs = np.array(diffs)
    stage = pd.DataFrame([
        dict(model="Model 1 (order placed)", pr_auc=average_precision_score(yb, both.m1), roc_auc=roc_auc_score(yb, both.m1),
             recall_top10=top_k(yb, both.m1.values, 0.10)["recall"]),
        dict(model=f"Model 3 (handoff, {CHOSEN_FAMILY})", pr_auc=average_precision_score(yb, both.m3), roc_auc=roc_auc_score(yb, both.m3),
             recall_top10=top_k(yb, both.m3.values, 0.10)["recall"])])
    print(f"\nSame {len(both):,} test orders:")
    display(stage.round(4))
    print(f"Model 3 − Model 1 PR-AUC: {diffs.mean():+.4f}  (95% interval {np.percentile(diffs, 2.5):+.4f} to {np.percentile(diffs, 97.5):+.4f})")
else:
    print("\n(model1_production_test_predictions.csv not uploaded — skipping the Model 1 vs Model 3 comparison)")

In [ ]:
# ---------------------------------------------------------------- save the production Model 3
bundle = {
    "model": d["model"], "model_name": f"model3_{CHOSEN_FAMILY}", "algorithm": CHOSEN_FAMILY, "scored_at": "handed to carrier",
    "features": FEATURES, "categorical": CATEGORICAL, "codes": codes,
    "threshold": d["threshold"], "fpr_target": FPR_TARGET, "calibrator": d["calibrator"],
    "config": CHOSEN_PARAMS, "weight_multiplier": WEIGHT_MULTIPLIER,
    "split": {"train": ["2017-01-01", "2018-04-01"], "calibration": ["2018-04-01", "2018-05-01"], "test": ["2018-05-01", "2018-09-01"]},
    "versions": VERSIONS, "calibration_metrics": cal_metrics, "test_metrics": test_metrics,
}
joblib.dump(bundle, "model3_production.joblib")

pd.DataFrame({"order_id": TEST.order_id, "actual_is_late": y_te, "raw_score": d["raw_te"],
              "late_probability": d["prob_te"], "alert": (d["raw_te"] >= d["threshold"]).astype(int)}
             ).to_csv("model3_production_test_predictions.csv", index=False)
cv.assign(params=cv.params.map(json.dumps)).to_csv("model3_cv_all_configs.csv", index=False)
report.to_csv("model3_family_comparison.csv", index=False)
json.dump({"chosen": CHOSEN_FAMILY, "params": CHOSEN_PARAMS, "features": FEATURES, "threshold": d["threshold"],
           "calibration": cal_metrics, "test": test_metrics, "versions": VERSIONS},
          open("model3_production_metrics.json", "w"), indent=2, default=float)

open("model3_inference.py", "w").write('''import joblib
import numpy as np
import pandas as pd

def load_bundle(path="model3_production.joblib"):
    return joblib.load(path)

def prepare_features(df, bundle):
    # df: rows of handed_to_carrier events (shipment_events_enriched.csv columns)
    X = df[bundle["features"]].copy()
    for c in bundle["categorical"]:
        X[c] = X[c].astype(str).map(bundle["codes"][c]).fillna(-1).astype(int)
    for c in bundle["features"]:
        if c not in bundle["categorical"]:
            X[c] = X[c].astype(float)
    return X

def score_orders(df, bundle):
    # Returns (raw_score, late_probability, alert), same contract as model1_inference.py
    raw = bundle["model"].predict_proba(prepare_features(df, bundle))[:, 1]
    return raw, bundle["calibrator"].predict(raw), raw >= bundle["threshold"]
''')

files_out = ["model3_production.joblib", "model3_inference.py", "model3_production_metrics.json",
             "model3_production_test_predictions.csv", "model3_cv_all_configs.csv", "model3_family_comparison.csv"]
with zipfile.ZipFile("Model3_Production.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_out:
        z.write(f)
print("Saved Model3_Production.zip:", files_out)
try:
    from google.colab import files
    files.download("Model3_Production.zip")
except Exception:
    pass